#  Student Productivity Clustering — DBSCAN

**Dataset:** `ultimate_student_productivity_dataset_5000.csv`  
**Task:** Unsupervised Clustering — Deteksi Kelompok & Outlier Mahasiswa  
**Model:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

print('Libraries loaded ')

---
## 1.  Cara Melihat Tipe Data

In [ ]:
df = pd.read_csv('../ultimate_student_productivity_dataset_5000.csv')
print(f'Shape: {df.shape}')
df.info()

---
## 2.  Dataset Bisa Digunakan Untuk Apa

**DBSCAN vs KMeans:**

| Aspek | KMeans | DBSCAN |
|-------|--------|--------|
| Jumlah cluster | Harus ditentukan | Otomatis ditemukan |
| Bentuk cluster | Hanya bulat (convex) | Bentuk apapun |
| Outlier | Semua titik masuk cluster | Titik noise = label -1 |
| Parameter | K | eps, min_samples |
| Kecepatan | Sangat cepat | Lebih lambat untuk data besar |

In [ ]:
drop_cols = ['student_id', 'productivity_score', 'exam_score']
df_proc = df.drop(columns=drop_cols).copy()
le = LabelEncoder()
for col in df_proc.select_dtypes(include='object').columns:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_proc)
print(f'Feature matrix: {X_scaled.shape}')

---
## 3.  Penjelasan DBSCAN

**Core concept:**
- Titik $p$ adalah **core point** jika ada ≥ `min_samples` titik dalam radius `eps`
- Titik yang dapat dicapai dari core point = **border point**
- Titik yang tidak bisa dicapai = **noise point** (label = -1)

$$N_{\varepsilon}(p) = \{q \in D \mid dist(p, q) \leq \varepsilon\}$$

**Kelebihan DBSCAN:**
- Tidak perlu menentukan K
- Tahan outlier
- Menemukan cluster berbentuk arbitrary

---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `eps` | 0.5 | Radius neighborhood ε |
| `min_samples` | 5 | Min titik untuk core point |
| `metric` | `'euclidean'` | Jarak: `'manhattan'`, `'cosine'`, dll |
| `algorithm` | `'auto'` | `'ball_tree'`, `'kd_tree'` untuk speedup |
| `n_jobs` | 1 | Paralel core |

**Tips praktis:**
- `eps` kecil → banyak noise, cluster kecil
- `eps` besar → sedikit cluster besar
- `min_samples` ≈ 2 × jumlah dimensi (aturan praktis)

---
## 5.  Evaluasi Yang Dipakai — Mencari eps Optimal

In [ ]:
# K-Distance Graph untuk menentukan eps
min_samples = max(5, X_scaled.shape[1])  # rule: 2 * n_dims, minimal 5
nn = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_dists = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(10, 5))
plt.plot(k_dists)
plt.xlabel('Titik data (diurutkan)'); plt.ylabel(f'{min_samples}-NN Distance')
plt.title('K-Distance Graph — Cari "siku" untuk eps')
plt.yscale('log'); plt.grid(True); plt.tight_layout(); plt.show()
print(f'Pakai min_samples={min_samples}. Lihat "siku" grafik di atas untuk menentukan eps.')

In [ ]:
# Scan eps values
eps_values = np.arange(1.0, 5.5, 0.5)
results = []
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
    labels = db.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    sil = silhouette_score(X_scaled[labels != -1], labels[labels != -1]) if n_clusters >= 2 and (labels != -1).sum() > 10 else np.nan
    results.append({'eps': eps, 'n_clusters': n_clusters, 'n_noise': n_noise, 'silhouette': sil})
    print(f'eps={eps:.1f}: clusters={n_clusters}, noise={n_noise}, sil={sil:.4f}' if not np.isnan(sil) else f'eps={eps:.1f}: clusters={n_clusters}, noise={n_noise}')

res_df = pd.DataFrame(results)
print(f'\nBest eps by silhouette: {res_df.loc[res_df["silhouette"].idxmax(), "eps"]}')

In [ ]:
# Fit final model
valid = res_df.dropna(subset=['silhouette'])
best_eps = float(valid.loc[valid['silhouette'].idxmax(), 'eps'])
print(f'Menggunakan eps={best_eps}, min_samples={min_samples}')

dbscan = DBSCAN(eps=best_eps, min_samples=min_samples, n_jobs=-1)
labels = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

df_proc['cluster'] = labels
df['cluster'] = labels
print(f'Jumlah cluster: {n_clusters}')
print(f'Noise points  : {n_noise} ({n_noise/len(labels)*100:.1f}%)')

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

In [ ]:
non_noise_mask = labels != -1
if non_noise_mask.sum() > 10 and n_clusters >= 2:
    sil = silhouette_score(X_scaled[non_noise_mask], labels[non_noise_mask])
    dbi = davies_bouldin_score(X_scaled[non_noise_mask], labels[non_noise_mask])
    print(f'Silhouette Score: {sil:.4f}')
    print(f'Davies-Bouldin  : {dbi:.4f}')
    if sil > 0.5: print(' Clustering bagus')
    elif sil > 0.3: print(' Clustering cukup, bisa ditingkatkan')
    else: print(' Clustering kurang baik, ubah eps / min_samples')
else:
    print('Tidak cukup cluster untuk evaluasi, coba eps yang lebih besar')

print(f'\nRingkasan cluster:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualisasi PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

unique_labels = sorted(set(labels))
colors = ['red' if c == -1 else sns.color_palette('tab10', n_clusters)[c] for c in unique_labels]

plt.figure(figsize=(10, 7))
for i, c in enumerate(unique_labels):
    mask = labels == c
    label_name = 'Noise/Outlier' if c == -1 else f'Cluster {c}'
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.5,
                color=colors[i], label=f'{label_name} (n={mask.sum()})')

plt.title(f'DBSCAN (eps={best_eps}, min_samples={min_samples}) — PCA 2D')
plt.legend(markerscale=3, loc='upper right'); plt.tight_layout(); plt.show()

---
## 7.  Cara Mengoptimasi Model — Profil Cluster & Outlier

In [ ]:
# Profil cluster (exclude noise)
numeric_cols = ['study_hours', 'sleep_hours', 'social_media_hours', 'mental_health_score',
                'focus_index', 'burnout_level', 'productivity_score']

profile_df = df[df['cluster'] != -1].groupby('cluster')[numeric_cols].mean().round(2)
print('Profil Cluster (mean):')
print(profile_df)

if n_noise > 0:
    print(f'\nProfil Outlier (n={n_noise}):')
    print(df[df['cluster'] == -1][numeric_cols].mean().round(2))

In [ ]:
# Pairplot untuk melihat distribusi beberapa fitur antar cluster
if n_clusters >= 2:
    sample = df.sample(min(500, len(df)), random_state=42)
    sns.pairplot(sample[['study_hours', 'sleep_hours', 'focus_index', 'burnout_level', 'cluster']],
                 hue='cluster', palette='tab10', plot_kws={'alpha': 0.4, 's': 10})
    plt.suptitle('Pairplot Fitur per Cluster (sampel)', y=1.02); plt.tight_layout(); plt.show()

---
## 8.  Cara Menyimpan Model

> **Catatan:** DBSCAN **tidak bisa memprediksi cluster untuk data baru** secara langsung (semi-supervised workaround diperlukan). Kita simpan parameter + scaler + label asli.
> Alternatif: gunakan `predict()` dari KNeighborsClassifier yang di-fit pada cluster hasil DBSCAN.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Latih KNN pada cluster label (exclude noise)
X_non_noise = X_scaled[non_noise_mask]
y_non_noise = labels[non_noise_mask]
knn_clf = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_clf.fit(X_non_noise, y_non_noise)

os.makedirs('saved_models', exist_ok=True)
joblib.dump(dbscan,     'saved_models/dbscan_productivity.pkl')
joblib.dump(scaler,     'saved_models/scaler_dbscan.pkl')
joblib.dump(knn_clf,    'saved_models/knn_for_dbscan_predict.pkl')
joblib.dump(list(df_proc.drop(columns=['cluster']).columns), 'saved_models/feature_cols_dbscan.pkl')
joblib.dump({'best_eps': best_eps, 'min_samples': min_samples, 'n_clusters': n_clusters}, 'saved_models/dbscan_params.pkl')
print(' DBSCAN + scaler + KNN predictor tersimpan!')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_sc     = joblib.load('saved_models/scaler_dbscan.pkl')
loaded_knn    = joblib.load('saved_models/knn_for_dbscan_predict.pkl')
feat_cols     = joblib.load('saved_models/feature_cols_dbscan.pkl')
dbscan_params = joblib.load('saved_models/dbscan_params.pkl')

print(f'Model dimuat ')
print(f'DBSCAN params: eps={dbscan_params["best_eps"]}, min_samples={dbscan_params["min_samples"]}')
print(f'Jumlah cluster: {dbscan_params["n_clusters"]}')

In [ ]:
mahasiswa_baru = pd.DataFrame([
    {'age': 20, 'gender': 1, 'academic_level': 1, 'study_hours': 6, 'self_study_hours': 3,
     'online_classes_hours': 2, 'social_media_hours': 1.5, 'gaming_hours': 0.5, 'sleep_hours': 7.5,
     'screen_time_hours': 4, 'exercise_minutes': 45, 'caffeine_intake_mg': 100,
     'part_time_job': 0, 'upcoming_deadline': 1, 'internet_quality': 2,
     'mental_health_score': 8, 'focus_index': 7, 'burnout_level': 2},
    {'age': 24, 'gender': 0, 'academic_level': 2, 'study_hours': 0.5, 'self_study_hours': 0,
     'online_classes_hours': 0, 'social_media_hours': 8, 'gaming_hours': 6, 'sleep_hours': 4,
     'screen_time_hours': 14, 'exercise_minutes': 0, 'caffeine_intake_mg': 400,
     'part_time_job': 1, 'upcoming_deadline': 0, 'internet_quality': 0,
     'mental_health_score': 2, 'focus_index': 1, 'burnout_level': 10}
])[feat_cols]

X_new_scaled = loaded_sc.transform(mahasiswa_baru)
cluster_pred = loaded_knn.predict(X_new_scaled)

for i, c in enumerate(cluster_pred):
    print(f'Mahasiswa {i+1}: Cluster {c}')
    if c in profile_df.index:
        print(f'  Profil cluster: study={profile_df.loc[c,"study_hours"]:.1f}h, '
              f'sleep={profile_df.loc[c,"sleep_hours"]:.1f}h, '
              f'productivity={profile_df.loc[c,"productivity_score"]:.1f}')

---
##  Perbandingan KMeans vs DBSCAN

| | KMeans | DBSCAN |
|--|--------|--------|
| **K/eps** | Harus ditentukan | eps ditentukan dengan k-distance graph |
| **Outlier** | Tidak dikenali | Label -1 (noise) |
| **Cluster shapes** | Hanya bulat | Arbitrary |
| **Scalability** | O(nKt) — sangat cepat | O(n log n) dengan index |
| **Predict baru** | `predict()` langsung | Perlu workaround (KNN) |
| **Metrik eval** | Silhouette, DBI | Silhouette, DBI, n_noise |